In [2]:
import pickle
from itertools import product

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector, Parameter
from qiskit.qasm3 import dump as qasm3_dump

from hubo_qaoa.utils.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian
from hubo_qaoa.utils.gfa_utils import gfa_file_to_graph
from hubo_qaoa.utils.parameterise_circuit import parameterise_circuit
from hubo_qaoa.utils.lr_qaoa import get_LR_qaoa_circuit

In [3]:
data_file = '/lustre/scratch127/qpg/jc59/new_hubo_formulation/circuit_depths/results.couplingall.precompute.0.pkl'
with open(data_file, 'rb') as f:
    res = pickle.load(f)

In [4]:
res.keys()

dict_keys(['test_N2_W2', 'trivial', 'test_N3_W4', 'test_N4_W5', 'test_N7_W2', 'test_N7_W3', 'test_N7_W4', 'test_N8_W2', 'test_N8_W3', 'test_N8_W5', 'test_N8_W6'])

In [5]:
res['test_N2_W2']['rzz']

{'count': 10,
 'depth': 9,
 'layers': 0,
 'layout': Layout({
 0: <Qubit register=(4, "q"), index=0>,
 1: <Qubit register=(4, "q"), index=1>,
 2: <Qubit register=(4, "q"), index=2>,
 3: <Qubit register=(4, "q"), index=3>
 }),
 'circuit': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x7f056bb562c0>}

In [11]:
data_file = '/lustre/scratch127/qpg/jc59/new_hubo_formulation/circuit_depths/results.couplingall.precompute.0.pkl'
with open(data_file, 'rb') as f:
    res = pickle.load(f)
    
delta_b_fixed = 0.75
delta_g_fixed = 0.30
for (filename, copy_numbers, name) in zip(
        [
            'test_N2_W2',
            'trivial',
            'test_N3_W4',
            'test_N7_W2',
            'test_N4_W5'
        ],
        [
            [1,1],
            [1,1,1],
            [2,1,1],
            [1,0,0,0,0,0,1],
            [2,1,1,1]
        ],
        [
            'I4',
            'I9',
            'I12',
            'I8',
            'I15'
        ]
    ):
    filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{filename}.gfa'
    graph, n, V, T = gfa_file_to_graph(filepath, copy_numbers)
    hamiltonian, norm = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)
    hamiltonian = hamiltonian * norm
    with open(f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/new_hubo_formulation/ionq/hubo_hamiltonian_{name}.pkl', 'wb') as f:
        pickle.dump(hamiltonian, f)

    cost_circuit = parameterise_circuit(res[filename]['rzz']['circuit'], parameter=Parameter('γ'))
    num_qubits: int = cost_circuit.num_qubits    
    print(filename, name, num_qubits)

    phis = ParameterVector('ϕ', num_qubits)
    for p in [1,2,3,4,5]:
        fixed_qc, circuit = get_LR_qaoa_circuit(p, delta_b_fixed, delta_g_fixed, num_qubits, cost_circuit, None, phis, True)
        with open(f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/new_hubo_formulation/ionq/qasm3/iter_qaoa_hubo_circuit_{name}_{p}.txt', 'w') as f:
            qasm3_dump(fixed_qc, f)

Keeping constraints at times: [0]
test_N2_W2 I4 4
16:43:38 - hubo_qaoa.utils.lr_qaoa - INFO - p = 1. Circuit depth: 15
16:43:38 - hubo_qaoa.utils.lr_qaoa - INFO - p = 2. Circuit depth: 28
16:43:38 - hubo_qaoa.utils.lr_qaoa - INFO - p = 3. Circuit depth: 41
16:43:38 - hubo_qaoa.utils.lr_qaoa - INFO - p = 4. Circuit depth: 54
16:43:38 - hubo_qaoa.utils.lr_qaoa - INFO - p = 5. Circuit depth: 67
Keeping constraints at times: [1 0]
trivial I9 9
16:43:39 - hubo_qaoa.utils.lr_qaoa - INFO - p = 1. Circuit depth: 121
16:43:39 - hubo_qaoa.utils.lr_qaoa - INFO - p = 2. Circuit depth: 240
16:43:39 - hubo_qaoa.utils.lr_qaoa - INFO - p = 3. Circuit depth: 359
16:43:39 - hubo_qaoa.utils.lr_qaoa - INFO - p = 4. Circuit depth: 478
16:43:39 - hubo_qaoa.utils.lr_qaoa - INFO - p = 5. Circuit depth: 597
Keeping constraints at times: [1 0 2]
test_N3_W4 I12 12
16:43:39 - hubo_qaoa.utils.lr_qaoa - INFO - p = 1. Circuit depth: 162
16:43:39 - hubo_qaoa.utils.lr_qaoa - INFO - p = 2. Circuit depth: 320
16:43:39 -

In [19]:
data_file = '/lustre/scratch127/qpg/jc59/new_hubo_formulation/circuit_depths/results.couplingall.precompute.0.pkl'
with open(data_file, 'rb') as f:
    res = pickle.load(f)
    
delta_b_fixed = 0.75
delta_g_fixed = 0.30
for (filename, copy_numbers, name) in zip(
        ('test_N4_W5', 'test_N7_W4', 'test_N8_W5', 'test_N8_W6'),
        ([2,1,1,1], [1,1,1,0,0,0,1], [1,1,1,1,0,0,0,1], [1,1,0,1,1,1,0,1],),
        [
            'I15',
            'I16',
            'I20',
            'I24',
        ]
    ):
    filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{filename}.gfa'
    graph, n, V, T = gfa_file_to_graph(filepath, copy_numbers)
    hamiltonian, norm = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)
    hamiltonian = hamiltonian * norm
    with open(f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/new_hubo_formulation/ionq/hubo_hamiltonian_{name}.pkl', 'wb') as f:
        pickle.dump(hamiltonian, f)

    cost_circuit = parameterise_circuit(res[filename]['rzz']['circuit'], parameter=Parameter('γ'))
    num_qubits: int = cost_circuit.num_qubits    
    print(filename, name, num_qubits)

    phis = ParameterVector('ϕ', num_qubits)
    for p in [1,2,3,4,5]:
        fixed_qc, circuit = get_LR_qaoa_circuit(p, delta_b_fixed, delta_g_fixed, num_qubits, cost_circuit, None, phis, True)
        with open(f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/new_hubo_formulation/ionq/qasm3/iter_qaoa_hubo_circuit_{name}_{p}.txt', 'w') as f:
            qasm3_dump(fixed_qc, f)

Keeping constraints at times: [0 1 3 2]
test_N4_W5 I15 15
16:46:24 - hubo_qaoa.utils.lr_qaoa - INFO - p = 1. Circuit depth: 232
16:46:24 - hubo_qaoa.utils.lr_qaoa - INFO - p = 2. Circuit depth: 460
16:46:24 - hubo_qaoa.utils.lr_qaoa - INFO - p = 3. Circuit depth: 688
16:46:25 - hubo_qaoa.utils.lr_qaoa - INFO - p = 4. Circuit depth: 916
16:46:25 - hubo_qaoa.utils.lr_qaoa - INFO - p = 5. Circuit depth: 1144
Keeping constraints at times: [0 2 1]
test_N7_W4 I16 16
16:46:25 - hubo_qaoa.utils.lr_qaoa - INFO - p = 1. Circuit depth: 800
16:46:25 - hubo_qaoa.utils.lr_qaoa - INFO - p = 2. Circuit depth: 1596
16:46:25 - hubo_qaoa.utils.lr_qaoa - INFO - p = 3. Circuit depth: 2392
16:46:26 - hubo_qaoa.utils.lr_qaoa - INFO - p = 4. Circuit depth: 3188
16:46:26 - hubo_qaoa.utils.lr_qaoa - INFO - p = 5. Circuit depth: 3984
Keeping constraints at times: [1 0 2 3]
test_N8_W5 I20 20
16:46:27 - hubo_qaoa.utils.lr_qaoa - INFO - p = 1. Circuit depth: 785
16:46:27 - hubo_qaoa.utils.lr_qaoa - INFO - p = 2. Ci

In [16]:
filename = 'test_N8_W5'
copy_numbers = [1,1,1,1,0,0,0,1]
cost_circuit = parameterise_circuit(res[filename]['rzz']['circuit'], parameter=Parameter('γ'))
filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{filename}.gfa'
graph, n, V, T = gfa_file_to_graph(filepath, copy_numbers)
hamiltonian, norm = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)

Keeping constraints at times: [0 2 3 1]


In [17]:
res[filename]['rzz']['layout']

Layout({
0: <Qubit register=(24, "q"), index=0>,
1: <Qubit register=(24, "q"), index=1>,
2: <Qubit register=(24, "q"), index=2>,
3: <Qubit register=(24, "q"), index=3>,
4: <Qubit register=(24, "q"), index=4>,
5: <Qubit register=(24, "q"), index=5>,
6: <Qubit register=(24, "q"), index=6>,
7: <Qubit register=(24, "q"), index=7>,
8: <Qubit register=(24, "q"), index=8>,
9: <Qubit register=(24, "q"), index=9>,
10: <Qubit register=(24, "q"), index=10>,
11: <Qubit register=(24, "q"), index=11>,
12: <Qubit register=(24, "q"), index=12>,
13: <Qubit register=(24, "q"), index=13>,
14: <Qubit register=(24, "q"), index=14>,
15: <Qubit register=(24, "q"), index=15>,
16: <Qubit register=(24, "q"), index=16>,
17: <Qubit register=(24, "q"), index=17>,
18: <Qubit register=(24, "q"), index=18>,
19: <Qubit register=(24, "q"), index=19>,
20: <Qubit register=(24, "q"), index=20>,
21: <Qubit register=(24, "q"), index=21>,
22: <Qubit register=(24, "q"), index=22>,
23: <Qubit register=(24, "q"), index=23>
})

In [18]:
hamiltonian.num_qubits

20

In [5]:
cost_circuit.draw(fold=-1)

┌───────────────┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               